In [3]:
import numpy #works = kernel connects

In [4]:
!which python

/lustre1/project/stg_00002/mambaforge/vsc35050/envs/pycistopic_v3_012025/bin/python


In [48]:
from pathlib import Path
import pandas as pd
import argparse
import os
from glob import glob

To start, we generate the sample to fragments path file. As `--suffix` you give the suffix of your fragment file as it is named in your folder. For us the fragment files are called `CNA_10xv2_1.FULL.fragments.tsv.gz` and `CNA_10xv2_2.FULL.fragments.tsv.gz` with CNA_10xv2_1 and CNA_10xv2_2 being our samples. 

In [26]:
!scatac_toolkit_utils map_sample_to_fragments \
  "/staging/leuven/stg_00002/lcb/teaching_regenomics/data" \
  "sample_to_fragments.tsv" \
  --suffix ".FULL.fragments.tsv.gz"


In [49]:
!head sample_to_fragments.tsv

sample	path_to_fragment_file
CNA_10xv2_1	/lustre1/project/stg_00002/lcb/teaching_regenomics/data/CNA_10xv2_1.FULL.fragments.tsv.gz
CNA_10xv2_2	/lustre1/project/stg_00002/lcb/teaching_regenomics/data/CNA_10xv2_2.FULL.fragments.tsv.gz


For annotation, we make use of an early generated annotation tsv file that contains the columns `barcode`,	`sample`,	`consensus_cell_type`. The barcode should be in the same format as in the fragment files, the sample should have the same labels as in the `sample_to_fragments.tsv` we generated earlier.

In [37]:
!scatac_toolkit_utils map_cell_names_to_all_cells \
  "/staging/leuven/stg_00002/lcb/teaching_regenomics/data/rna_cell_types_annot_split.tsv" \
  "all_cells_with_annotation.tsv"


In [50]:
!head all_cells_with_annotation.tsv

sample	cell_type	cell_barcode
CNA_10xv2_1	CD4+ T cell	AAACAACGAAAGCTAA
CNA_10xv2_1	CD4+ T cell	AAACAACGAAGACAAT
CNA_10xv2_1	CD4+ T cell	AAACAACGAAGCCATT
CNA_10xv2_1	CD14+ monocyte	AAACAACGAAGGCATG
CNA_10xv2_1	CD16+ monocyte	AAACAACGAAGGTGAC
CNA_10xv2_1	B cell	AAACAACGAAGTCGGA
CNA_10xv2_1	CD4+ T cell	AAACAACGAAGTGGTG
CNA_10xv2_1	CD4+ T cell	AAACAACGAATCCTTT
CNA_10xv2_1	CD4+ T cell	AAACAACGAATGTGGG


As you use the full fragment files but only a subset of the cells are annotated (the ones who passed the QC earlier), you have to filter the 

In [39]:
!scatac_toolkit_utils filter_sample_to_fragments \
  sample_to_fragments.tsv \
  all_cells_with_annotation.tsv \
  sample_to_fragments.filtered.tsv

Wrote: /lustre1/project/stg_00002/lcb/jdeman/software/pycisTopic/notebooks/sample_to_fragments.filtered.tsv
Samples in filtered fragments: ['CNA_10xv2_1']
Samples in annotations: ['CNA_10xv2_1']


In [ ]:
!head sample_to_fragments.filtered.tsv

now run sc toolkit itself (first generate fragment files per cell type)

In [40]:
!scatac_fragment_tools split \
    -f sample_to_fragments.filtered.tsv \
    -b all_cells_with_annotation.tsv \
    -c /staging/leuven/res_00001/genomes/homo_sapiens/hg38_ucsc/fasta/hg38.chrom.sizes \
    -t /tmp \
    -o fragments

Using the .fragments.ts.gz files, you can generate a consensus peaks file to start the tutorial of pyCisTopic. 

Alternatively, if you already have done this and generated the all_cells_with_annotation.tsv based on you analysed adata instead of an imported annotation file, you can continue to generate BigWigs as in the second part of this tutorial: 

Now we generate BigWigs for each cell type from its respective fragment file we just generated in out output directory (`-o`): `fragments`.

In [43]:
!mkdir fragments/bw
!scatac_fragment_tools bigwig \
    -i fragments/B_cell.fragments.tsv.gz \
    -c /staging/leuven/res_00001/genomes/homo_sapiens/hg38_ucsc/fasta/hg38.chrom.sizes \
    -o fragments/bw/B_cell.bw

for additional options, such as writing a cut site instead of a coverage BigWig, check out `https://aertslab.github.io/scatac_fragment_tools/bigwig.html`

If you want to generate BigWigs for all the cell types in parallel, write a `bigwig_commands.txt` similar to what will be done/has been done in the QC part of pyCisTopic.

In [46]:
# Inputs
fragments_dir = "fragments"
chrom_sizes = "/staging/leuven/res_00001/genomes/homo_sapiens/hg38_ucsc/fasta/hg38.chrom.sizes"
output_dir = "fragments/bw"

bigwig_commands_filename = "bigwig_commands.txt"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Collect fragment files
fragment_files = sorted(glob(os.path.join(fragments_dir, "*.fragments.tsv.gz")))

# Write BigWig commands
with open(bigwig_commands_filename, "w") as fh:
    for fragment_filename in fragment_files:
        # Extract cell type / sample name
        base = os.path.basename(fragment_filename)
        sample = base.replace(".fragments.tsv.gz", "")

        bigwig_filename = os.path.join(output_dir, f"{sample}.bw")

        print(
            "scatac_fragment_tools bigwig",
            f"-i {fragment_filename}",
            f"-c {chrom_sizes}",
            f"-o {bigwig_filename}",
            "--normalize", #add more commands from scatac_fragment_tools bigwig here if you'd like (e.g., -x to generate cutsite BigWigs instead of coverage ones as default)
            sep=" ",
            file=fh,
        )

print(f"Wrote {len(fragment_files)} commands to {bigwig_commands_filename}")

Wrote 7 commands to bigwig_commands.txt


In [47]:
!cat bigwig_commands.txt | parallel -j 4

Done! Now you can use the BigWig files to train e.g., CREsted models aas described in `https://crested.readthedocs.io/en/stable/`.